# Archived guarded VMD source audit

Diagnose the failed 17.9272 dB replay. No cache relaxation, model fitting, parameter search, new VMD or regional inference occurs. Legacy unknown-montage evidence remains separate from the verified frontal campaign.

In [ ]:
import os, sys, tempfile, subprocess, json, hashlib, zipfile
from pathlib import Path
PHASE = 'snr-guard'
COMMIT = '0b183e6c09f6ecfbb3ed4818028eb5abbdf842fc' or os.environ.get('EOG_COMMIT', 'research/region-aware-eog-kaggle')

IN_KAGGLE = Path("/kaggle/input").exists()
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
ENVIRONMENT = "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "local"
if not IN_KAGGLE and os.environ.get("EOG_ALLOW_NON_KAGGLE") != "1":
    raise RuntimeError("Kaggle-only campaign; portable execution needs explicit EOG_ALLOW_NON_KAGGLE=1")
TEMP_ROOT = Path(tempfile.mkdtemp(prefix="eog-run-"))
CHECKOUT = TEMP_ROOT / "repository"
subprocess.check_call(["git", "clone", "--quiet", "https://github.com/saltypal/Lateral-eye-Artifact-removal.git", str(CHECKOUT)])
subprocess.check_call(["git", "-C", str(CHECKOUT), "checkout", "--quiet", COMMIT])
GIT_SHA = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
ENV_ROOT = TEMP_ROOT / "environment"
subprocess.check_call([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(ENV_ROOT)])
ENV_PY = ENV_ROOT / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
# Kaggle's OS Python omits ensurepip; host pip can install into a pip-free venv.
subprocess.check_call([sys.executable, "-m", "pip", "--python", str(ENV_PY), "install", "--quiet", "--progress-bar", "off", "-r", str(CHECKOUT / "requirements-research.txt")])
PROCESS_ENV = os.environ.copy()
PROCESS_ENV["PYTHONPATH"] = str(CHECKOUT)
PROCESS_ENV["PYTHONUNBUFFERED"] = "1"
hardware = subprocess.check_output([str(ENV_PY), "-c", "import torch,json; print(json.dumps(dict(device='cuda:0' if torch.cuda.is_available() else 'cpu',gpu_count=torch.cuda.device_count())))"], text=True, env=PROCESS_ENV)
print("Environment", ENVIRONMENT, "Commit", GIT_SHA, "Hardware", hardware.strip())
if os.environ.get("TPU_NAME") or os.environ.get("COLAB_TPU_ADDR"):
    print("TPU detected. PyTorch-XLA is unvalidated; explicit CPU path applies without CUDA.")
ROOT_WORK = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("EOG_PERSISTENT_RESULTS", "./eog-results")).resolve()
RUN_SPEC = None  # injected by guard17_replay.py
if RUN_SPEC is None:
    RUN_SPEC = {'run_id': 'guard17-manual', 'stage': 'guard17-replay', 'campaign_id': 'vmd-bandroute-20261010', 'git_sha': '0b183e6c09f6ecfbb3ed4818028eb5abbdf842fc', 'orchestrator_git_sha': 'manual archived reproduction', 'recipe': {'campaign_id': 'vmd-bandroute-20261010', 'stage': 'guard17-replay', 'legacy_git_sha': '0b183e6c09f6ecfbb3ed4818028eb5abbdf842fc', 'source_checkpoint_sha256': '4bb7c1b0cfa0e8341c4053ab3099efb16832f045f99e37afde1904a93c2cd0a9', 'source_checkpoint_git_sha': '2ec0b3cd3da636cc3aed3b476ff890a6f56e2627', 'K': 3, 'alpha': 2000, 'model': 'EOGContextGainStudent', 'epoch': 110, 'identity_weight': 0.2, 'learning_rate': 0.001, 'inference_gate_mode': 'hard', 'inference_gate_offset': 0.0, 'strength': 1.0, 'expected_historical_snr_db': 17.927211250454842, 'role': 'archived development reproduction; not fresh confirmation or current teacher qualification', 'kernel_sources': ['satyapaladugu/region-aware-eog-benchmark/2', 'satyapaladugu/region-aware-eog-restore/1', 'satyapaladugu/region-aware-eog-vmd-engine/1', 'satyapaladugu/region-aware-eog-reference-safe/1', 'satyapaladugu/region-aware-eog-snr-context/1'], 'dataset_sources': ['satyapaladugu/lateral-eye-complete-dataset'], 'enable_gpu': False, 'reason_cpu': 'Frozen-weight inference and cached-mode reproduction; no retraining and no dependency on the stalled GPU session'}, 'fresh_confirmation': False, 'teacher_qualified': False}
if GIT_SHA != RUN_SPEC["git_sha"]:
    raise RuntimeError("Archived source SHA mismatch")
OUT = ROOT_WORK / "results" / RUN_SPEC["campaign_id"] / RUN_SPEC["run_id"]
OUT.mkdir(parents=True, exist_ok=False)
(OUT/"run_spec.json").write_text(json.dumps(RUN_SPEC,indent=2))
(OUT/"execution_state.json").write_text(json.dumps({"status":"running","run_id":RUN_SPEC["run_id"]}))
print("Archived replay",RUN_SPEC["run_id"],GIT_SHA,flush=True)

## Legacy numerical fixtures and source verification

In [ ]:
test = subprocess.run([str(ENV_PY), "-m", "pytest", str(CHECKOUT / "tests"), "-q"], cwd=CHECKOUT,
                      env=PROCESS_ENV, capture_output=True, text=True)
(OUT / "contract_tests.txt").write_text(test.stdout + test.stderr)
print(test.stdout)
print(test.stderr)
if test.returncode:
    raise RuntimeError("Kaggle numeric contract tests failed")

## Verified original data and restoration

In [ ]:
if IN_KAGGLE:
    manifests = list(Path("/kaggle/input").rglob("source_manifest.json"))
    if len(manifests) != 1:
        raise RuntimeError("Attach exactly one complete research dataset")
    MANIFEST = manifests[0]
    attached = MANIFEST.parent
    payload = json.loads(MANIFEST.read_text())
    archives = list(attached.glob("*.eogbundle")) + list(attached.glob("*.zip"))
    if archives:
        container = payload.get("archive")
        if container:
            print("Verifying outer archive", container["name"], flush=True)
            expected_archive = attached / container["name"]
            digest = hashlib.sha256()
            with expected_archive.open("rb") as handle:
                for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
                    digest.update(block)
            if expected_archive.stat().st_size != container["bytes"] or digest.hexdigest() != container["sha256"]:
                raise RuntimeError("Outer research archive SHA256 mismatch")
        extraction = TEMP_ROOT / "data"
        extraction.mkdir()
        print("Extracting complete research input", flush=True)
        for archive_path in archives:
            with zipfile.ZipFile(archive_path) as archive:
                for member in archive.infolist():
                    target = (extraction / member.filename).resolve()
                    if not target.is_relative_to(extraction.resolve()):
                        raise ValueError("Unsafe ZIP path")
                archive.extractall(extraction)
        attached = extraction
    candidates = list(attached.rglob("klados_contaminated_eeg.npy"))
    if len(candidates) != 1:
        raise RuntimeError("Missing or ambiguous Klados dataset")
    DATA_ROOT = candidates[0].parent.parent
else:
    DATA_ROOT = Path(os.environ["EOG_DATA_ROOT"]).resolve()
    MANIFEST = Path(os.environ["EOG_SOURCE_MANIFEST"]).resolve()
supplements = list(Path("/kaggle/input").rglob("supplement_manifest.json")) if IN_KAGGLE else []
if len(supplements) > 1:
    raise RuntimeError("Attach at most one verified OSF restoration output")
if PHASE in {"train", "calibration", "neural-search", "vmd-convergence", "vmd-robust-grid", "reference-guided", "reference-refine", "reference-safe", "snr-target", "snr-context", "snr-guard"} and len(supplements) != 1:
    raise RuntimeError("This phase requires the completed study04 restoration output")
if supplements:
    subprocess.check_call([str(ENV_PY), "-m", "eog_vmd_fcm_bgru.bundle_io", "--root", str(DATA_ROOT),
                           "--manifest", str(supplements[0]), "--output", str(OUT / "attached_supplement.json")],
                          cwd=CHECKOUT, env=PROCESS_ENV)
print("Input", DATA_ROOT, "Output", OUT)

## Compare cache samples against current preprocessing
The Git module saves exact-match status, relative error, waveform correlation, checkpoint/source digests and versions. A high correlation alone does not authorize replay.

In [ ]:
IMPLEMENTATION = TEMP_ROOT / 'audit-implementation'
subprocess.check_call(['git','clone','--quiet','https://github.com/saltypal/Lateral-eye-Artifact-removal.git',str(IMPLEMENTATION)])
subprocess.check_call(['git','-C',str(IMPLEMENTATION),'checkout','--quiet',RUN_SPEC['orchestrator_git_sha']])
actual = subprocess.check_output(['git','-C',str(IMPLEMENTATION),'rev-parse','HEAD'],text=True).strip()
if actual != RUN_SPEC['orchestrator_git_sha']:
    raise RuntimeError('Audit implementation identity differs')
command = [str(ENV_PY),str(IMPLEMENTATION/'src/vmd_eog/legacy_guard_audit.py'),
           '--root',str(DATA_ROOT),'--output',str(OUT),
           '--checkpoint-sha256',RUN_SPEC['recipe']['source_checkpoint_sha256']]
result = subprocess.run(command,cwd=CHECKOUT,env=PROCESS_ENV)
status = 'complete' if result.returncode == 0 else 'failed'
manifest = {str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest()
            for p in OUT.rglob('*') if p.is_file() and p.name not in ('execution_state.json','artifact_manifest.json')}
(OUT/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2))
(OUT/'execution_state.json').write_text(json.dumps({'status':status,'run_id':RUN_SPEC['run_id'],
    'git_sha':GIT_SHA,'orchestrator_git_sha':actual,'fresh_confirmation':False,'teacher_qualified':False},indent=2))
if result.returncode:
    raise RuntimeError('Archived source audit failed; preserve logs and partial evidence')
print((OUT/'guard17_source_audit.json').read_text())
